In [ ]:
# Please run the command then scroll down to upload the file WBES_data with the button
# ============================================================
from google.colab import files
uploaded = files.upload()
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_excel("WBES_data.xlsx", sheet_name="Recoded")

df["group"] = np.select(
    [(df["b2b"] >= 95) & (df["b2b"] <= 100),
     (df["b2b"] >= 20) & (df["b2b"] <= 90)],
    ["WOS", "JV"],
    default=None
)

df = df[df["group"].notna()]
print(df["group"].value_counts())

# ============================================================
def mannwhitney_compare(data, var, group_col="group", group_a="WOS", group_b="JV"):
    """
    Compares one variable between two groups using the Mann-Whitney U test.

    Returns median, n, U, p-value, and rank-biserial r (effect size)
    for both groups.
    """
    a = data.loc[data[group_col] == group_a, var].dropna()
    b = data.loc[data[group_col] == group_b, var].dropna()

    u_stat, p_value = stats.mannwhitneyu(a, b, alternative="two-sided")

    # rank-biserial r: 1 - 2U / (n1*n2)
    # positive r -> group_a tends to have higher values
    r = 1 - (2 * u_stat) / (len(a) * len(b))

    return {
        "variable": var,
        f"n_{group_a}": len(a),
        f"n_{group_b}": len(b),
        f"median_{group_a}": a.median(),
        f"median_{group_b}": b.median(),
        f"mean_{group_a}": a.mean(),
        f"mean_{group_b}": b.mean(),
        "U": u_stat,
        "p_value": p_value,
        "rank_biserial_r": round(r, 3),
    }

# ============================================================
severity_vars = ["j30f", "j7a", "j31", "h30", "j2", "j30c", "e30", "d30b", "j30a"]

results = pd.DataFrame([mannwhitney_compare(df, v) for v in severity_vars])
results = results.sort_values("p_value").reset_index(drop=True)
results

# ============================================================
#BH correction

from statsmodels.stats.multitest import multipletests

# reject: True/False per test at alpha=0.05 after correction
# p_BH: the BH-adjusted p-values
reject, p_bh, _, _ = multipletests(results["p_value"], alpha=0.05, method="fdr_bh")

results["p_BH"] = p_bh.round(4)
results["significant_BH"] = reject

results